# Day 05: Testing the star-analysis business logic

These tests exercise the calculations and validation used by the Day 04 HR-diagram app, independently of its web routes. They use Python's built-in `unittest` framework and a small in-memory catalog fixture; the original CSV is not changed.

## Load the application module

The notebook finds `app.py` relative to the current working directory, so it can be run from this folder or from the workspace. Importing the app also loads the real catalog once, as it does when the website starts.

In [ ]:
from pathlib import Path
import sys
import unittest
from unittest.mock import patch

import pandas as pd

working_directories = (Path.cwd(), *Path.cwd().parents)
app_directory = None
for directory in working_directories:
    candidates = (
        directory,
        directory / "Day04-goingOnline",
        directory / "Day03-assignment" / "Day04-goingOnline",
    )
    app_directory = next(
        (candidate for candidate in candidates if (candidate / "app.py").is_file()),
        None,
    )
    if app_directory is not None:
        break

if app_directory is None:
    raise FileNotFoundError("Could not find Day04-goingOnline/app.py from the current directory.")

sys.path.insert(0, str(app_directory))
import app as star_app

print(f"Testing application: {app_directory / 'app.py'}")

## Define business-logic tests

The catalog tests substitute controlled rows for the CSV reader so the expected calculations are known. Parsing tests cover accepted values and common invalid inputs. The closest-star test checks that luminosity distance is measured logarithmically, matching the HR diagram.

In [ ]:
class CatalogCalculationTests(unittest.TestCase):
    def test_catalog_derives_distance_luminosity_and_temperature(self):
        catalog = pd.DataFrame(
            {
                "id": ["known-star"],
                "parallax_arcsec": [0.5],
                "flux_Lsuncm2": [1e-20],
                "peakLam_m": [5e-7],
            }
        )

        with patch.object(star_app.pd, "read_csv", return_value=catalog):
            result = star_app.load_catalog()

        self.assertEqual(len(result), 1)
        star = result.iloc[0]
        expected_distance_pc = 1 / 0.5
        expected_distance_cm = expected_distance_pc * star_app.PARSEC_TO_CM
        expected_luminosity = 4 * star_app.np.pi * expected_distance_cm**2 * 1e-20
        expected_temperature = star_app.WIEN_DISPLACEMENT_M_K / 5e-7
        self.assertAlmostEqual(star["distance_pc"], expected_distance_pc)
        self.assertAlmostEqual(star["luminosity_Lsun"], expected_luminosity)
        self.assertAlmostEqual(star["temperature_K"], expected_temperature)

    def test_catalog_drops_invalid_measurements(self):
        catalog = pd.DataFrame(
            {
                "id": ["valid", "zero-parallax", "bad-flux", "zero-wavelength"],
                "parallax_arcsec": [0.5, 0, 0.25, 0.25],
                "flux_Lsuncm2": [1e-20, 1e-20, "not-a-number", 1e-20],
                "peakLam_m": [5e-7, 5e-7, 6e-7, 0],
            }
        )

        with patch.object(star_app.pd, "read_csv", return_value=catalog):
            result = star_app.load_catalog()

        self.assertEqual(result["id"].tolist(), ["valid"])

    def test_catalog_reports_missing_required_columns(self):
        catalog = pd.DataFrame({"id": ["incomplete"], "parallax_arcsec": [0.5]})

        with patch.object(star_app.pd, "read_csv", return_value=catalog):
            with self.assertRaisesRegex(ValueError, "Catalog is missing required columns"):
                star_app.load_catalog()


class MeasurementParsingTests(unittest.TestCase):
    def test_valid_inputs_convert_exponent_to_solar_luminosity(self):
        temperature, luminosity = star_app.parse_measurements(
            {"temperature_K": " 5778 ", "luminosity_exponent": "-2"}
        )

        self.assertEqual(temperature, 5778)
        self.assertAlmostEqual(luminosity, 0.01)

    def test_inclusive_diagram_boundaries_are_accepted(self):
        lower = star_app.parse_measurements(
            {"temperature_K": "2500", "luminosity_exponent": "-4"}
        )
        upper = star_app.parse_measurements(
            {"temperature_K": "15000", "luminosity_exponent": "4"}
        )

        self.assertEqual(lower, (2500, 1e-4))
        self.assertEqual(upper, (15000, 1e4))

    def test_missing_values_are_rejected(self):
        with self.assertRaisesRegex(ValueError, "Enter both temperature and luminosity"):
            star_app.parse_measurements({"temperature_K": "5000", "luminosity_exponent": ""})

    def test_fractional_exponent_is_rejected(self):
        with self.assertRaisesRegex(ValueError, "whole-number luminosity exponent"):
            star_app.parse_measurements(
                {"temperature_K": "5000", "luminosity_exponent": "-2.5"}
            )

    def test_non_finite_or_out_of_range_temperature_is_rejected(self):
        for temperature in ("nan", "inf", "2499", "15001"):
            with self.subTest(temperature=temperature):
                with self.assertRaisesRegex(ValueError, "Temperature must be between"):
                    star_app.parse_measurements(
                        {"temperature_K": temperature, "luminosity_exponent": "0"}
                    )

    def test_out_of_range_luminosity_exponent_is_rejected(self):
        with self.assertRaisesRegex(ValueError, "Luminosity exponent must be between"):
            star_app.parse_measurements(
                {"temperature_K": "5000", "luminosity_exponent": "5"}
            )


class ClosestCatalogStarTests(unittest.TestCase):
    def test_match_uses_normalized_temperature_and_log_luminosity(self):
        samples = pd.DataFrame(
            {
                "id": ["same-temperature-bright", "warmer-same-luminosity"],
                "temperature_K": [7500, 10000],
                "luminosity_Lsun": [1e4, 1],
            },
            index=[10, 20],
        )

        with patch.object(star_app, "VISIBLE_CATALOG", samples):
            closest = star_app.closest_catalog_star(7500, 1)

        self.assertEqual(closest["id"], "warmer-same-luminosity")
        self.assertEqual(closest.name, 20)


suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("One or more star-analysis business-logic tests failed.")

## Understanding the result

A passing run reports **10 tests** with `OK`. The tests verify intermediate derived values as well as accepted and rejected inputs, and they make the catalog-match coordinate rule explicit. They do not test Flask response rendering; that remains covered by the Day 04 `test_app.py` suite.